<a href="https://colab.research.google.com/github/DeepthiManthapuram/Deep_Learning/blob/main/Attention_mechanism.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Attention Mechanism**

In [20]:
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras.layers import(
    Input,
    LSTM,
    Embedding,
    Dense,
    Concatenate,
    Dot,
    Softmax,
    Attention
)

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

#Prepare Dataset

In [7]:
english_sentences = [
"i love ai",
"i love deep learning",
"how are you",
"good morning"]

french_sentences = [
"start j aime ai end",
"start j aime apprentissage profond end",
"start comment allez vous end",
"start bonjour end"]

# Tokenization
Neural networks understand numbers, not text.

In [9]:
eng_tokenizer = Tokenizer()
eng_tokenizer.fit_on_texts(english_sentences)

fra_tokenizer = Tokenizer()
fra_tokenizer.fit_on_texts(french_sentences)

print("English vocabulary:")
print(eng_tokenizer.word_index)

print("French vocabulary:")
print(fra_tokenizer.word_index)

English vocabulary:
{'i': 1, 'love': 2, 'ai': 3, 'deep': 4, 'learning': 5, 'how': 6, 'are': 7, 'you': 8, 'good': 9, 'morning': 10}
French vocabulary:
{'start': 1, 'end': 2, 'j': 3, 'aime': 4, 'ai': 5, 'apprentissage': 6, 'profond': 7, 'comment': 8, 'allez': 9, 'vous': 10, 'bonjour': 11}


# Convert sentences into Sequences

In [11]:
encoder_input  = eng_tokenizer.texts_to_sequences(english_sentences)
decoder_input  = fra_tokenizer.texts_to_sequences(french_sentences)

print("English sequences:")
print(encoder_input)
print("French sequences:")
print(decoder_input)

English sequences:
[[1, 2, 3], [1, 2, 4, 5], [6, 7, 8], [9, 10]]
French sequences:
[[1, 3, 4, 5, 2], [1, 3, 4, 6, 7, 2], [1, 8, 9, 10, 2], [1, 11, 2]]


# Padding
Padding makes all sequences equal length

In [12]:
encoder_input = pad_sequences(
    encoder_input,
    padding='post'
)

decoder_input = pad_sequences(
    decoder_input,
    padding='post'
)

print("English sequences:")
print(encoder_input)
print("French sequences:")
print(decoder_input)

English sequences:
[[ 1  2  3  0]
 [ 1  2  4  5]
 [ 6  7  8  0]
 [ 9 10  0  0]]
French sequences:
[[ 1  3  4  5  2  0]
 [ 1  3  4  6  7  2]
 [ 1  8  9 10  2  0]
 [ 1 11  2  0  0  0]]


# Build Encoder
Encoder generates hidden states for every word

In [13]:
encoder_inputs = Input(shape=(None,))
encoder_embedding = Embedding(
    input_dim = len(eng_tokenizer.word_index)+1,
    output_dim = 64
)(encoder_inputs)

encoder_outputs, state_h, state_c = LSTM(
    units=64,
    return_sequences = True,
    return_state = True
)(encoder_embedding)

print("Encoder created successfully")

Encoder created successfully


# Build Decoder

In [16]:
decoder_inputs = Input(shape=(None,))
decoder_embedding = Embedding(
    input_dim = len(eng_tokenizer.word_index)+1,
    output_dim = 64
)(decoder_inputs)

decoder_outputs, state_h, state_c = LSTM(
    units=64,
    return_sequences = True,
    return_state = True
)(decoder_embedding)

print("Decoder created successfully")

Decoder created successfully


# Attention Layer

In [25]:
attention=Attention()
attention_output = attention([decoder_outputs, encoder_outputs])

print("Attention layer created successfully")

Attention layer created successfully


# Combine Decoder + Attention Output

In [26]:
decoder_combined = Concatenate(axis=-1)([decoder_outputs, attention_output])
print("Decoder + Attention combined successfully")

Decoder + Attention combined successfully


# Final prediction layer

In [27]:
output=Dense(
len(fra_tokenizer.word_index)+1,
activation='softmax'
)(decoder_combined)

print("Prediction Layer Created Successfully")

Prediction Layer Created Successfully


In [28]:
model = Model(inputs=[encoder_inputs, decoder_inputs], outputs=output)
print("Model created successfully")

Model created successfully


# Compile Model

In [32]:
model.compile(
    optimizer='adam',
    loss = 'sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Model compiled")

Model compiled


# Train Model

In [31]:
decoder_target = np.expand_dims(decoder_input, -1)

model.fit(
    [encoder_input, decoder_input],
    decoder_target,
    epochs=20,
    batch_size=2
)

Epoch 1/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 6s 41ms/step - accuracy: 0.1250 - loss: 2.4827
Epoch 2/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.2917 - loss: 2.4706
Epoch 3/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.3750 - loss: 2.4609
Epoch 4/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.3750 - loss: 2.4482 
Epoch 5/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.4167 - loss: 2.4363
Epoch 6/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.4167 - loss: 2.4229
Epoch 7/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.3750 - loss: 2.4099
Epoch 8/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.3750 - loss: 2.3916 
Epoch 9/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.3750 - loss: 2.3729
Epoch 10/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.3333 - loss: 2.3518
Epoch 11/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.3333 - loss: 2.3272
Epoch 12/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - accuracy: 0.3333 - loss: 2.2982

In [47]:
print(model.history.history['accuracy'][-1])

0.3333333432674408


In [48]:
def predict_translation(sentence):

    seq = eng_tokenizer.texts_to_sequences([sentence])

    seq = pad_sequences(
        seq,
        maxlen=encoder_input.shape[1],
        padding="post"
    )

    decoder_seq = np.zeros((1, decoder_input.shape[1]))
    decoder_seq[0,0] = fra_tokenizer.word_index['start']

    prediction = model.predict(
        [seq, decoder_seq],
        verbose=0
    )

    print("Predicted IDs:")

    for timestep in prediction[0]:
        print(np.argmax(timestep), end=" ")

    print()

In [51]:
predict_translation("i love ai")

Predicted IDs:
1 0 0 0 0 0 


In [52]:
print("English :", "i love ai")
print("French  :", predict_translation("i love ai"))

print()

print("English :", "how are you")
print("French  :", predict_translation("how are you"))

print()

print("English :", "good morning")
print("French  :", predict_translation("good morning"))

English : i love ai
Predicted IDs:
1 0 0 0 0 0 
French  : None

English : how are you
Predicted IDs:
1 0 0 0 0 0 
French  : None

English : good morning
Predicted IDs:
0 0 0 0 0 0 
French  : None
